# Retrieval Strategy Evaluation with Ground-Truth Metrics

Uses `data/eval_dataset.jsonl` to compute standard IR metrics across 4 retrieval strategies:
- **Recall@K**: Was the ground-truth chunk in the top K?
- **Precision@K**: What fraction of retrieved chunks are relevant?
- **MRR**: Average reciprocal rank of first relevant result
- **NDCG@K**: Rank-weighted relevance quality
- **Hit Rate@K**: At least one relevant chunk in top K?

**Prerequisite:** Run `eval_dataset_generation.ipynb` first to produce `data/eval_dataset.jsonl`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, "..")
load_dotenv()

# Load eval dataset
EVAL_PATH = Path("../data/eval_dataset.jsonl")
if not EVAL_PATH.exists():
    raise FileNotFoundError(f"Eval dataset not found at {EVAL_PATH}. Run eval_dataset_generation.ipynb first.")

eval_data = []
with open(EVAL_PATH) as f:
    for line in f:
        eval_data.append(json.loads(line))

print(f"✓ Loaded {len(eval_data)} evaluation QA pairs")

# Preview
for qa in eval_data[:3]:
    print(f"\n  [{qa['metadata']['question_type']}] {qa['question']}")
    print(f"  Answer: {qa['reference_answer'][:80]}...")

In [ ]:
from db.database import session_scope
from db.schema import Chunk
from ingestion.embed import VectorStoreConfig, VectorStoreManager
from langchain_core.documents import Document as LCDocument
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Connect to PGVector
config = VectorStoreConfig()
vsm = VectorStoreManager(config)
vector_store = vsm.vector_store
print(f"✓ Connected to PGVector collection: {config.collection_name}")

# Load all chunks for BM25
with session_scope() as session:
    db_chunks = session.query(Chunk).all()
    chunk_docs = [
        LCDocument(
            page_content=c.chunk_text,
            metadata={"uuid": c.uuid, "document_id": c.document_id, **(c.chunk_metadata or {})},
        )
        for c in db_chunks
    ]

bm25_retriever = BM25Retriever.from_documents(chunk_docs, k=5)
print(f"✓ Built BM25 index over {len(chunk_docs)} chunks")

In [ ]:
def retrieve_topk(query: str, k: int = 5) -> list[dict]:
    """Strategy 1: Top-K similarity search."""
    results = vector_store.similarity_search_with_score(query, k=k)
    return [
        {"uuid": doc.metadata.get("uuid", ""), "score": float(score), "text": doc.page_content}
        for doc, score in results
    ]

def retrieve_mmr(query: str, k: int = 5) -> list[dict]:
    """Strategy 2: MMR (Maximal Marginal Relevance)."""
    results = vector_store.max_marginal_relevance_search(query, k=k, fetch_k=20)
    return [
        {"uuid": doc.metadata.get("uuid", ""), "score": None, "text": doc.page_content}
        for doc in results
    ]

def retrieve_threshold(query: str, k: int = 5, threshold: float = 0.7) -> list[dict]:
    """Strategy 3: Similarity score threshold (only return chunks above threshold)."""
    results = vector_store.similarity_search_with_score(query, k=k)
    return [
        {"uuid": doc.metadata.get("uuid", ""), "score": float(score), "text": doc.page_content}
        for doc, score in results
        if float(score) >= threshold
    ]

def retrieve_hybrid(query: str, k: int = 5) -> list[dict]:
    """Strategy 4: Hybrid BM25 + Semantic (EnsembleRetriever with RRF)."""
    semantic_retriever = vector_store.as_retriever(search_kwargs={"k": k})
    ensemble = EnsembleRetriever(
        retrievers=[bm25_retriever, semantic_retriever],
        weights=[0.4, 0.6],
    )
    results = ensemble.invoke(query)[:k]
    return [
        {"uuid": doc.metadata.get("uuid", ""), "score": None, "text": doc.page_content}
        for doc in results
    ]

strategies = {
    "Top-K": retrieve_topk,
    "MMR": retrieve_mmr,
    "Threshold": retrieve_threshold,
    "Hybrid (BM25+Semantic)": retrieve_hybrid,
}

print(f"✓ Defined {len(strategies)} retrieval strategies")

In [ ]:
import time

all_results = {}  # {strategy_name: [per-query result dicts]}

for strategy_name, retrieve_fn in strategies.items():
    print(f"\nRunning: {strategy_name}")
    strategy_results = []

    for qa in eval_data:
        retrieved = retrieve_fn(qa["question"])
        strategy_results.append({
            "eval_id": qa["id"],
            "question": qa["question"],
            "question_type": qa["metadata"]["question_type"],
            "ground_truth_uuids": set(qa["chunk_ids"]),
            "retrieved_uuids": [r["uuid"] for r in retrieved],
            "n_retrieved": len(retrieved),
        })

    all_results[strategy_name] = strategy_results
    print(f"  ✓ {len(strategy_results)} queries processed")

print(f"\n✓ All strategies complete")

In [ ]:
def compute_metrics(results: list[dict], k: int = 5) -> dict:
    """Compute IR metrics for a list of query results."""
    recalls = []
    precisions = []
    reciprocal_ranks = []
    ndcgs = []
    hits = []

    for r in results:
        gt = r["ground_truth_uuids"]
        retrieved = r["retrieved_uuids"][:k]

        # Recall@K: fraction of ground truth found in top K
        found = sum(1 for uuid in retrieved if uuid in gt)
        recall = found / len(gt) if gt else 0
        recalls.append(recall)

        # Precision@K: fraction of retrieved that are relevant
        precision = found / len(retrieved) if retrieved else 0
        precisions.append(precision)

        # Hit Rate@K: binary — at least one hit
        hit = 1 if found > 0 else 0
        hits.append(hit)

        # MRR: reciprocal rank of first relevant result
        rr = 0
        for rank, uuid in enumerate(retrieved, 1):
            if uuid in gt:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

        # NDCG@K
        dcg = 0
        for rank, uuid in enumerate(retrieved, 1):
            rel = 1 if uuid in gt else 0
            dcg += rel / np.log2(rank + 1)
        # Ideal DCG: all relevant docs at top
        ideal_rels = sorted([1] * min(len(gt), k) + [0] * max(0, k - len(gt)), reverse=True)
        idcg = sum(rel / np.log2(rank + 2) for rank, rel in enumerate(ideal_rels))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcgs.append(ndcg)

    return {
        f"Recall@{k}": np.mean(recalls),
        f"Precision@{k}": np.mean(precisions),
        "MRR": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcgs),
        f"Hit Rate@{k}": np.mean(hits),
    }

# Compute for all strategies
metrics_table = {}
for strategy_name, results in all_results.items():
    metrics_table[strategy_name] = compute_metrics(results, k=5)

metrics_df = pd.DataFrame(metrics_table).T
metrics_df = metrics_df.round(4)
print("=== Strategy Comparison ===")
print(metrics_df.to_string())

In [ ]:
print("=== Per-Question-Type Breakdown ===\n")

for strategy_name, results in all_results.items():
    print(f"\n--- {strategy_name} ---")
    by_type = {}
    for r in results:
        qt = r["question_type"]
        if qt not in by_type:
            by_type[qt] = []
        by_type[qt].append(r)

    type_metrics = {}
    for qt, qt_results in sorted(by_type.items()):
        m = compute_metrics(qt_results, k=5)
        type_metrics[qt] = m

    type_df = pd.DataFrame(type_metrics).T
    print(type_df.round(4).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Retrieval Strategy Comparison", fontsize=14, fontweight="bold")

colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

for idx, metric in enumerate(metrics_df.columns):
    ax = axes[idx]
    bars = ax.bar(range(len(metrics_df)), metrics_df[metric], color=colors)
    ax.set_title(metric, fontsize=11)
    ax.set_xticks(range(len(metrics_df)))
    ax.set_xticklabels(metrics_df.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)

    # Add value labels on bars
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../data/retrieval_eval_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved to data/retrieval_eval_comparison.png")

In [ ]:
# Hit Rate@5 by strategy × question type heatmap
heatmap_data = {}
for strategy_name, results in all_results.items():
    by_type = {}
    for r in results:
        qt = r["question_type"]
        if qt not in by_type:
            by_type[qt] = []
        by_type[qt].append(r)

    row = {}
    for qt, qt_results in sorted(by_type.items()):
        m = compute_metrics(qt_results, k=5)
        row[qt] = m["Hit Rate@5"]
    heatmap_data[strategy_name] = row

heatmap_df = pd.DataFrame(heatmap_data).T
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_df.values, cmap="YlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)

# Add text annotations
for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        val = heatmap_df.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val > 0.7 else "black", fontsize=10)

ax.set_title("Hit Rate@5 by Strategy × Question Type")
plt.colorbar(im, ax=ax, label="Hit Rate")
plt.tight_layout()
plt.savefig("../data/retrieval_eval_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved to data/retrieval_eval_heatmap.png")